In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[3]))
from utils.helpers import load_data

# Detailed analysis of time during the exam. expexted time vs. time taken task by task + by gender. see if time taken by task affects performance during each task.

## 1. Data Loading

In [48]:
df = load_data("../data/cleaned_data/IN1020_2024_cleaned_data.csv")
df.head()

Data loaded successfully from ../data/cleaned_data/IN1020_2024_cleaned_data.csv


,exam_start_time,exam_end_time,extra_time_mins,incident_time_mins,gender,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,candidate_response_code,candidate_response_text,total_score
0,2024-12-11 14:00:02,2024-12-11 15:57:52,0,0,M,418,2.4,1.1,Tallsystemer,85,2.4,simpleChoice_1368096164456,Heksadesimale tall,71.61
1,2024-12-11 14:00:02,2024-12-11 15:57:52,0,0,M,418,2.4,1.1,Tallsystemer,85,2.4,simpleChoice_IA172399520449584402464-f255-46e6...,Oktale tall,71.61
2,2024-12-11 14:00:02,2024-12-11 15:57:52,0,0,M,418,2.4,1.1,Tallsystemer,85,2.4,simpleChoice_IA17239952044953d28e157-ac4c-49db...,Titallsystemet,71.61
3,2024-12-11 14:00:02,2024-12-11 15:57:52,0,0,M,418,2.4,1.1,Tallsystemer,85,2.4,simpleChoice_IA1723995204495960f601e-73d5-48f8...,Heksadesimale tall,71.61
4,2024-12-11 14:00:02,2024-12-11 15:57:52,0,0,M,418,2.4,1.1,Tallsystemer,85,2.4,simpleChoice_IA17239952636882cb13f25-bd73-4352...,Titallsystemet,71.61


## 2. Data Preparation

In [49]:
df_time = df.groupby(['student_id', 'question_number']).agg({
    'gender': 'first',
    'max_question_score': 'first',
    'auto_score_per_question': 'first',
    'question_duration_sec': 'first',
    'exam_start_time': 'first',
    'exam_end_time': 'first',
    'extra_time_mins': 'first',
    'incident_time_mins': 'first'
}).reset_index()

df_time['time_taken_exam_min'] = round((pd.to_datetime(df_time['exam_end_time']) - pd.to_datetime(df_time['exam_start_time'])).dt.total_seconds() / 60, 2)
df_time = df_time.drop(columns=['exam_start_time', 'exam_end_time'])

df_time['expected_time_per_question_sec'] = 144 * df_time['max_question_score']  # Assuming 144 sec (2.4 minutes) per point since total exam time is 240 minutes for 100 points

df_time.head()

,student_id,question_number,gender,max_question_score,auto_score_per_question,question_duration_sec,extra_time_mins,incident_time_mins,time_taken_exam_min,expected_time_per_question_sec
0,1,1.1,M,2.4,2.4,52,0,0,136.97,345.6
1,1,1.2,M,3.0,3.0,105,0,0,136.97,432.0
2,1,1.3,M,3.8,0.0,328,0,0,136.97,547.2
3,1,1.4,M,3.0,3.0,118,0,0,136.97,432.0
4,1,1.5,M,3.0,3.0,69,0,0,136.97,432.0


## 3. Data exploration

In [52]:
extra_time_students_data = df_time[(df_time['extra_time_mins'] != 0) | (df_time['incident_time_mins'] != 0)]

extra_time_students_count = extra_time_students_data.groupby('student_id').agg({
    'extra_time_mins': 'first',
    'incident_time_mins': 'first'
}).reset_index()

counter1 = 0
counter2 = 0

for i in extra_time_students_count['extra_time_mins']:
    if i != 0:
        counter1 += 1

for i in extra_time_students_count['incident_time_mins']:
    if i != 0:
        counter2 += 1

print(f"There are {counter1} students with extra_time_mins (given before the exam).")
print(f"There are {counter2} students with incident_time_mins (given during the exam).")

There are 53 students with extra_time_mins (given before the exam).
There are 6 students with incident_time_mins (given during the exam).


In [53]:
# remove students with extra time or incident time
df_time = df_time[(df_time['extra_time_mins'] == 0) & (df_time['incident_time_mins'] == 0)]

In [54]:
# description and exploration of spesific columns in df_time
df_time[['time_taken_exam_min']].describe()

,time_taken_exam_min
count,15390.000000
mean,197.775536
std,43.918686
min,40.750000
25%,169.580000
50%,211.730000
75%,237.220000
max,240.770000


In [55]:
# description and exploration of spesific columns in df_time
df_time[['expected_time_per_question_sec', 'question_duration_sec']].describe()

,expected_time_per_question_sec,question_duration_sec
count,15390.000000,15390.000000
mean,480.000000,384.518129
std,164.834867,370.490344
min,144.000000,0.000000
25%,432.000000,145.000000
50%,432.000000,276.000000
75%,576.000000,501.000000
max,864.000000,7545.000000
